# KYC setup – schema, tables, and UC functions

**Run this notebook once** (or when you change catalog/schema) before using the KYC agent. It:
1. Creates the catalog (if not exists), schema, and the agent volume if they don't exist.
2. Creates and populates tables: `pep_screening`, `sanctions_screening`, `sanctioned_countries`, `fraud_list`, `accounts`, `sbi_risk_codes` (seed data so SME risk check works).
3. Creates the Unity Catalog functions: `check_pep`, `check_sanctions`, `check_sanctioned_country`, `check_customer_risk`, `check_fraud`.

SBI risk codes are filled from a **static table of ~100 CBS SBI codes** (no Excel required).

**Config:** Set the **catalog** and **schema** widgets at the top of the notebook. When called via `%run` from `02_kyc_agent` or `03_kyc_agent_deploy`, the parent's values are used automatically.

In [ ]:
%run ../_shared/config

In [ ]:
import os
from pyspark.sql.types import StructType, StructField, LongType, StringType, IntegerType
from pyspark.sql import Row

# CATALOG, SCHEMA, CATALOG_SCHEMA, VOLUME_PATH come from %run ../_shared/config
LLM_ENDPOINT = os.getenv("LLM_ENDPOINT", "databricks-meta-llama-3-3-70b-instruct")

print(f"Using CATALOG_SCHEMA={CATALOG_SCHEMA}, VOLUME_PATH={VOLUME_PATH}")

## 1. Create schema and tables

In [ ]:
# Create catalog and schema if they don't exist (CATALOG_SCHEMA e.g. main.kyc_demo)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_SCHEMA}")

# Create the agent volume if it doesn't exist (path e.g. /Volumes/main/kyc_demo/agent_volume)
vol_parts = VOLUME_PATH.strip("/").split("/")
if len(vol_parts) >= 4 and vol_parts[0].lower() == "volumes":
    _cat, _schema, _vol = vol_parts[1], vol_parts[2], vol_parts[3]
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {_cat}.{_schema}.{_vol}")
    print(f"Volume {_cat}.{_schema}.{_vol} ensured.")

# Mock PEP (Politically Exposed Persons) data
pep_schema = StructType([
    StructField("id", LongType()),
    StructField("first_name", StringType()),
    StructField("last_name", StringType()),
    StructField("country", StringType()),
    StructField("date_of_birth", StringType()),
    StructField("position", StringType()),
    StructField("list_source", StringType()),
])
pep_rows = [
    Row(1, "Jan", "de Vries", "Netherlands", "1975-03-22", "Mayor", "National PEP"),
    Row(2, "Maria", "Garcia", "Spain", "1968-07-10", "Minister", "EU PEP"),
    Row(3, "John", "Smith", "United Kingdom", "1982-11-30", "MP", "UK PEP"),
    Row(4, "Erik", "van den Berg", "Netherlands", "1970-05-14", "Senator", "National PEP"),
]
spark.createDataFrame(pep_rows, pep_schema).write.mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.pep_screening")

# Mock Sanctions data (no overlap with PEP)
sanctions_schema = StructType([
    StructField("id", LongType()),
    StructField("first_name", StringType()),
    StructField("last_name", StringType()),
    StructField("country", StringType()),
    StructField("date_of_birth", StringType()),
    StructField("sanction_type", StringType()),
    StructField("list_name", StringType()),
])
sanctions_rows = [
    Row(1, "Vladimir", "Petrov", "Russia", "1960-01-15", "Asset freeze", "OFAC"),
    Row(2, "Dimitri", "Kozlov", "Russia", "1972-08-20", "Travel ban", "EU Consolidated"),
    Row(3, "Anna", "Müller", "Germany", "1985-04-12", "Asset freeze", "UN List"),
    Row(4, "Pierre", "Dubois", "France", "1978-06-11", "Advisory", "National Watchlist"),
    Row(5, "Carlos", "Santos", "Brazil", "1965-09-05", "Asset freeze", "OFAC"),
]
spark.createDataFrame(sanctions_rows, sanctions_schema).write.mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.sanctions_screening")

# Sanctioned countries list (OFAC, EU, UN comprehensive sanctions programmes)
sanctioned_countries_schema = StructType([
    StructField("country", StringType()),
    StructField("sanction_programme", StringType()),
    StructField("list_source", StringType()),
])
sanctioned_countries_rows = [
    Row("North Korea", "Comprehensive", "OFAC / EU / UN"),
    Row("Iran", "Comprehensive", "OFAC / EU / UN"),
    Row("Syria", "Comprehensive", "OFAC / EU"),
    Row("Cuba", "Comprehensive", "OFAC"),
    Row("Russia", "Comprehensive", "OFAC / EU"),
    Row("Belarus", "Targeted", "EU"),
    Row("Myanmar", "Targeted", "OFAC / EU"),
    Row("Venezuela", "Targeted", "OFAC"),
    Row("Afghanistan", "Targeted", "OFAC / UN"),
    Row("Somalia", "Targeted", "OFAC / UN"),
    Row("Yemen", "Targeted", "OFAC / UN"),
    Row("Libya", "Targeted", "UN"),
    Row("South Sudan", "Targeted", "EU / UN"),
]
spark.createDataFrame(sanctioned_countries_rows, sanctioned_countries_schema).write.mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.sanctioned_countries")

# Mock shared fraud blacklist (customers and employees; multiple financial institutions)
fraud_schema = StructType([
    StructField("first_name", StringType()),
    StructField("last_name", StringType()),
    StructField("risk_score", IntegerType()),
    StructField("list_source", StringType()),
    StructField("role", StringType()),
])
fraud_rows = [
    Row("Jan", "Fraudster", 90, "Bank A", "customer"),
    Row("Maria", "Risky", 70, "Insurer B", "employee"),
    Row("Hans", "Betrüger", 85, "Bank C", "customer"),
    Row("Elena", "Kovac", 65, "Insurer A", "employee"),
]
spark.createDataFrame(fraud_rows, fraud_schema).write.mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.fraud_list")

# Accounts table (populated by create_account tool in kyc_agent)
spark.sql(f"""
  CREATE TABLE IF NOT EXISTS {CATALOG_SCHEMA}.accounts (
    id STRING,
    account_type STRING,
    account_user STRING
  )
""")

# SBI risk codes: static table of ~100 CBS SBI codes (Standaard Bedrijfsindeling); risk_level for KYC SME check
# Source: CBS SBI structure (https://www.cbs.nl/nl-nl/onze-diensten/methoden/classificaties/activiteiten/standaard-bedrijfsindeling--sbi--/sbi-code-zoeken)
sbi_schema = StructType([
    StructField("sbi_code", StringType()),
    StructField("risk_level", StringType()),
    StructField("description", StringType()),
])
SBI_STATIC = [
    ("01", "low", "Crop and animal production, hunting"),
    ("01.11", "low", "Growing of cereals, legumes, oil seeds"),
    ("01.13", "low", "Growing of vegetables, melons, roots"),
    ("01.21", "low", "Growing of grapes"),
    ("01.41", "low", "Raising of dairy cattle"),
    ("01.49", "low", "Raising of other cattle"),
    ("02", "low", "Forestry and logging"),
    ("02.10", "low", "Silviculture and other forestry activities"),
    ("03", "low", "Fishing and aquaculture"),
    ("03.11", "low", "Marine fishing"),
    ("05", "medium", "Mining of coal and lignite"),
    ("06", "medium", "Extraction of crude petroleum and natural gas"),
    ("07", "medium", "Mining of metal ores"),
    ("08", "medium", "Other mining and quarrying"),
    ("10", "low", "Manufacture of food products"),
    ("10.11", "low", "Processing and preserving of meat"),
    ("10.12", "low", "Processing and preserving of poultry"),
    ("10.39", "low", "Processing of fruit and vegetables"),
    ("10.71", "low", "Manufacture of bread and fresh pastry"),
    ("11", "low", "Manufacture of beverages"),
    ("13", "low", "Manufacture of textiles"),
    ("14", "low", "Manufacture of wearing apparel"),
    ("15", "low", "Manufacture of leather and related products"),
    ("16", "low", "Manufacture of wood and wood products"),
    ("17", "low", "Manufacture of paper and paper products"),
    ("20", "low", "Manufacture of chemicals and chemical products"),
    ("21", "low", "Manufacture of pharmaceuticals"),
    ("22", "low", "Manufacture of rubber and plastic products"),
    ("23", "low", "Manufacture of non-metallic mineral products"),
    ("24", "low", "Manufacture of basic metals"),
    ("25", "low", "Manufacture of fabricated metal products"),
    ("26", "low", "Manufacture of computer, electronic and optical products"),
    ("27", "low", "Manufacture of electrical equipment"),
    ("28", "low", "Manufacture of machinery and equipment n.e.c."),
    ("29", "low", "Manufacture of motor vehicles and trailers"),
    ("30", "low", "Manufacture of other transport equipment"),
    ("31", "low", "Manufacture of furniture"),
    ("32", "low", "Other manufacturing"),
    ("33", "low", "Repair and installation of machinery"),
    ("35", "medium", "Electricity, gas, steam and air conditioning supply"),
    ("36", "low", "Water collection, treatment and supply"),
    ("37", "low", "Sewerage"),
    ("38", "low", "Waste collection, treatment and disposal"),
    ("39", "low", "Remediation and other waste management"),
    ("41", "low", "Construction of buildings"),
    ("42", "low", "Civil engineering"),
    ("43", "low", "Specialised construction activities"),
    ("45", "medium", "Wholesale and retail trade and repair of motor vehicles"),
    ("46", "medium", "Wholesale trade, except of motor vehicles"),
    ("46.31", "medium", "Wholesale of fruit and vegetables"),
    ("46.52", "medium", "Wholesale of electronic and telecommunications equipment"),
    ("47", "medium", "Retail trade, except of motor vehicles and motorcycles"),
    ("47.11", "medium", "Retail sale in non-specialised stores (food, beverages, tobacco)"),
    ("47.19", "medium", "Other retail sale in non-specialised stores"),
    ("47.24", "medium", "Retail sale of bread, pastry, confectionery"),
    ("47.41", "medium", "Retail sale of computers, peripheral units, software"),
    ("47.91", "medium", "Retail sale via mail order or Internet"),
    ("49", "medium", "Land transport and transport via pipelines"),
    ("49.41", "medium", "Freight transport by road"),
    ("50", "medium", "Water transport"),
    ("51", "medium", "Air transport"),
    ("52", "medium", "Warehousing and support activities for transportation"),
    ("53", "medium", "Postal and courier activities"),
    ("55", "medium", "Accommodation"),
    ("56", "medium", "Food and beverage service activities"),
    ("56.10", "medium", "Restaurants and mobile food service"),
    ("56.21", "medium", "Event catering"),
    ("58", "low", "Publishing activities"),
    ("59", "low", "Motion picture, video and television programme production"),
    ("60", "low", "Broadcasting and programming"),
    ("61", "low", "Telecommunications"),
    ("62", "low", "Computer programming, consultancy and related"),
    ("63", "low", "Information service activities"),
    ("64", "high", "Financial service activities, except insurance and pension funding"),
    ("64.11", "high", "Central banking"),
    ("64.19", "high", "Other monetary intermediation"),
    ("64.20", "high", "Activities of holding companies"),
    ("64.91", "high", "Financial leasing"),
    ("64.92", "high", "Other credit granting"),
    ("64.99", "high", "Other financial service activities n.e.c."),
    ("65", "high", "Insurance, reinsurance and pension funding"),
    ("65.11", "high", "Life insurance"),
    ("65.12", "high", "Non-life insurance"),
    ("65.20", "high", "Reinsurance"),
    ("66", "high", "Activities auxiliary to financial services and insurance"),
    ("66.11", "high", "Administration of financial markets"),
    ("66.19", "high", "Other activities auxiliary to financial services"),
    ("68", "medium", "Real estate activities"),
    ("68.20", "medium", "Renting and operating of own or leased real estate"),
    ("69", "low", "Legal and accounting activities"),
    ("70", "low", "Activities of head offices; management consultancy"),
    ("71", "low", "Architectural and engineering activities"),
    ("72", "low", "Scientific research and development"),
    ("73", "low", "Advertising and market research"),
    ("74", "low", "Other professional, scientific and technical activities"),
    ("77", "low", "Rental and leasing activities"),
    ("78", "low", "Employment activities"),
    ("79", "low", "Travel agency and tour operator activities"),
    ("80", "low", "Security and investigation activities"),
    ("81", "low", "Services to buildings and landscape activities"),
    ("82", "low", "Office administrative and business support activities"),
    ("84", "low", "Public administration and defence; compulsory social security"),
    ("85", "low", "Education"),
    ("85.10", "low", "Pre-primary education"),
    ("85.20", "low", "Primary education"),
    ("85.31", "low", "General secondary education"),
    ("85.59", "low", "Other education n.e.c."),
    ("86", "low", "Human health activities"),
    ("86.10", "low", "Hospital activities"),
    ("86.90", "low", "Other human health activities"),
    ("87", "low", "Residential care activities"),
    ("88", "low", "Social work activities without accommodation"),
    ("90", "low", "Creative, arts and entertainment activities"),
    ("91", "low", "Libraries, archives, museums and other cultural activities"),
    ("92", "high", "Gambling and betting activities"),
    ("93", "low", "Sports activities and amusement and recreation activities"),
    ("94", "low", "Activities of membership organisations"),
    ("95", "low", "Repair of computers and personal and household goods"),
    ("96", "low", "Other personal service activities"),
    ("97", "low", "Activities of households as employers"),
    ("99", "low", "Activities of extraterritorial organisations and bodies"),
]
sbi_rows = [Row(code, level, desc) for code, level, desc in SBI_STATIC]
spark.createDataFrame(sbi_rows, sbi_schema).write.mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.sbi_risk_codes")

print(f"Created schema and tables in {CATALOG_SCHEMA}: pep_screening, sanctions_screening, sanctioned_countries, fraud_list, accounts, sbi_risk_codes ({len(SBI_STATIC)} CBS SBI codes)")

## 1b. Copy bundled files to volume (optional)

If you added files to the **volume_assets** folder in the bundle, run the cell below once to copy them into the agent volume so the KYC agent can use them (e.g. `passport_dirk_van_dijk.png`, `utility_bill_dirk_van_dijk.png`). Files follow the naming convention `passport_<first_name>_<last_name>.png` and `utility_bill_<first_name>_<last_name>.png`.

In [ ]:
# Copy files from bundle's volume_assets/ into the agent volume (Workspace API + Files API, no local disk)
import base64
import io
import os
try:
    from databricks.sdk import WorkspaceClient
    from databricks.sdk.service.workspace import ExportFormat

    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    parts = notebook_path.rstrip("/").split("/")
    nb_idx = next((i for i, p in enumerate(parts) if p == "notebooks"), None)
    if nb_idx is not None and nb_idx > 0:
        repo_root = "/".join(parts[:nb_idx])
        assets_path = repo_root + "/volume_assets"
        w = WorkspaceClient()
        try:
            for obj in w.workspace.list(assets_path):
                if getattr(obj.object_type, "value", obj.object_type) == "FILE":
                    name = os.path.basename(obj.path.rstrip("/"))
                    if name.startswith(".") or name.lower().endswith(".md"):
                        continue
                    dest = VOLUME_PATH.rstrip("/") + "/" + name
                    try:
                        resp = w.workspace.export(path=obj.path, format=ExportFormat.SOURCE)
                        content = resp.content
                        if content:
                            raw = base64.b64decode(content) if isinstance(content, str) else content
                            w.files.upload(dest, io.BytesIO(raw))
                            print(f"Copied {name} to volume.")
                    except Exception as ex:
                        print(f"Skip {name}: {ex}")
        except Exception as e:
            print(f"No volume_assets or copy skipped: {e}")
    else:
        print("Could not resolve repo root from notebook path; skip or upload files to the volume manually.")
except ImportError:
    print("Install databricks-sdk for Workspace API (pip install databricks-sdk). Copy skipped.")
except Exception as e:
    print(f"Copy bundled files skipped: {e}")

## 2. Create Unity Catalog functions

In [ ]:
# UC functions defined here (parameterized by CATALOG_SCHEMA); no external SQL file needed
cs = CATALOG_SCHEMA

check_pep_sql = f"""
CREATE OR REPLACE FUNCTION {cs}.check_pep(
  first_name STRING COMMENT 'First name from parsed KYC',
  last_name STRING COMMENT 'Last name from parsed KYC',
  country STRING COMMENT 'Country from parsed KYC',
  date_of_birth STRING COMMENT 'Date of birth from parsed KYC'
)
RETURNS TABLE (id LONG, first_name STRING, last_name STRING, country STRING, date_of_birth STRING, position STRING, list_source STRING)
COMMENT 'Check a person against the PEP list. Uses fuzzy name matching.'
RETURN
  SELECT p.id, p.first_name, p.last_name, p.country, p.date_of_birth, p.position, p.list_source
  FROM {cs}.pep_screening p
  WHERE levenshtein(lower(check_pep.first_name), lower(p.first_name)) <= 2
    AND levenshtein(lower(check_pep.last_name), lower(p.last_name)) <= 2
"""

check_sanctions_sql = f"""
CREATE OR REPLACE FUNCTION {cs}.check_sanctions(
  first_name STRING COMMENT 'First name from parsed KYC',
  last_name STRING COMMENT 'Last name from parsed KYC',
  country STRING COMMENT 'Country from parsed KYC',
  date_of_birth STRING COMMENT 'Date of birth from parsed KYC'
)
RETURNS TABLE (id LONG, first_name STRING, last_name STRING, country STRING, date_of_birth STRING, sanction_type STRING, list_name STRING)
COMMENT 'Check a person against the sanctions list. Uses fuzzy name matching.'
RETURN
  SELECT s.id, s.first_name, s.last_name, s.country, s.date_of_birth, s.sanction_type, s.list_name
  FROM {cs}.sanctions_screening s
  WHERE levenshtein(lower(check_sanctions.first_name), lower(s.first_name)) <= 2
    AND levenshtein(lower(check_sanctions.last_name), lower(s.last_name)) <= 2
"""

check_sanctioned_country_sql = f"""
CREATE OR REPLACE FUNCTION {cs}.check_sanctioned_country(
  country STRING COMMENT 'Country from parsed KYC (passport nationality)'
)
RETURNS TABLE (country STRING, sanction_programme STRING, list_source STRING)
COMMENT 'Check whether a country appears on the sanctioned countries list (OFAC, EU, UN). Uses case-insensitive exact match.'
RETURN
  SELECT sc.country, sc.sanction_programme, sc.list_source
  FROM {cs}.sanctioned_countries sc
  WHERE lower(trim(check_sanctioned_country.country)) = lower(trim(sc.country))
"""

check_customer_risk_sql = f"""
CREATE OR REPLACE FUNCTION {cs}.check_customer_risk(sbi_code STRING COMMENT 'SBI code of the business (e.g. 47.91)')
RETURNS TABLE (sbi_code STRING, risk_level STRING, description STRING)
COMMENT 'Check business SBI code against risk SBI codes. Valid range 01110-99000. If invalid or not in table, returns risk_level=failed.'
RETURN
  WITH input AS (
    SELECT trim(check_customer_risk.sbi_code) AS raw_code,
           try_cast(regexp_replace(regexp_replace(trim(check_customer_risk.sbi_code), '[.]', ''), ' ', '') AS INT) AS num_code
  ),
  valid_range AS (SELECT raw_code FROM input WHERE num_code BETWEEN 1110 AND 99000),
  matches AS (
    SELECT r.sbi_code, r.risk_level, r.description
    FROM {cs}.sbi_risk_codes r
    CROSS JOIN input i
    WHERE i.raw_code = r.sbi_code OR (length(r.sbi_code) <= length(i.raw_code) AND startswith(i.raw_code, r.sbi_code))
  )
  SELECT sbi_code, risk_level, description FROM matches
  UNION ALL
  SELECT i.raw_code, 'failed', 'SBI code not in risk table' FROM input i
  WHERE NOT EXISTS (SELECT 1 FROM matches) AND EXISTS (SELECT 1 FROM valid_range v WHERE v.raw_code = i.raw_code)
  UNION ALL
  SELECT i.raw_code, 'failed', 'SBI code out of valid range (01110-99000)' FROM input i
  WHERE NOT EXISTS (SELECT 1 FROM valid_range v WHERE v.raw_code = i.raw_code)
"""

check_fraud_sql = f"""
CREATE OR REPLACE FUNCTION {cs}.check_fraud(
  first_name STRING COMMENT 'First name from KYC',
  last_name STRING COMMENT 'Last name from KYC'
)
RETURNS TABLE (fraud_score INT, match_count INT, list_sources STRING)
COMMENT 'Check person against the shared fraud blacklist. Returns fraud_score, match_count, list_sources.'
RETURN
  SELECT coalesce(max(f.risk_score), 0) AS fraud_score, count(*) AS match_count,
         concat_ws(', ', collect_set(f.list_source)) AS list_sources
  FROM {cs}.fraud_list f
  WHERE levenshtein(lower(check_fraud.first_name), lower(f.first_name)) <= 2
    AND levenshtein(lower(check_fraud.last_name), lower(f.last_name)) <= 2
"""

for stmt in [check_pep_sql, check_sanctions_sql, check_sanctioned_country_sql, check_customer_risk_sql, check_fraud_sql]:
    spark.sql(stmt)
print(f"Created 5 UC functions in {CATALOG_SCHEMA}: check_pep, check_sanctions, check_sanctioned_country, check_customer_risk, check_fraud")

## 4. Grant read-only access to all users

Grants `SELECT`, `EXECUTE`, and `READ VOLUME` on the schema to `account users` — covers all current and future tables, functions, and volumes. Useful when publishing the demo to a shared workspace.

In [ ]:
# Read-only grants for all users on the demo schema.
# Uses `account users` group (account-level, present in every Databricks workspace).
for stmt in [
    f"GRANT USE SCHEMA ON SCHEMA {CATALOG_SCHEMA} TO `account users`",
    f"GRANT SELECT ON SCHEMA {CATALOG_SCHEMA} TO `account users`",
    f"GRANT EXECUTE ON SCHEMA {CATALOG_SCHEMA} TO `account users`",
    f"GRANT READ VOLUME ON SCHEMA {CATALOG_SCHEMA} TO `account users`",
]:
    spark.sql(stmt)
print(f"Granted read-only access on {CATALOG_SCHEMA} to `account users`")